# Kopi IoT Web3 — Generator model_config.json (Website / HF Space)

Memilih **model terbaik per variabel** (best-per-variable) lalu menghitung prediksi 14 hari
untuk disajikan website lewat HF Space.

**Konsisten dengan notebook Deep Learning:** evaluasi **multi-step (recursive)** untuk
ketiga model yang dapat di-deploy — **Naive, Linear (recursive-lag), Prophet** —
sehingga pemilihan mapping selaras dengan analisis multi-step.

Output: `model_config.json` (upload ke HF Space, menimpa yang lama).

Jalankan: Runtime -> Run all.

In [ ]:
!pip -q install prophet

## 1. Ambil & bersihkan data

In [ ]:
import requests, pandas as pd, numpy as np, json

CHANNEL_ID = 1848413
url = f'https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json'
feeds = requests.get(url, params={'results':8000,'timezone':'Asia/Jakarta'}, timeout=30).json()['feeds']

df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
if df['created_at'].dt.tz is not None: df['created_at'] = df['created_at'].dt.tz_localize(None)
for col,name in [('field1','suhu'),('field2','udara'),('field3','tanah')]:
    df[name] = pd.to_numeric(df[col], errors='coerce')
df = df[['created_at','suhu','udara','tanah']].set_index('created_at')

RANGES = {'suhu':(10,45),'udara':(0,100),'tanah':(0,100)}
for v,(lo,hi) in RANGES.items(): df[v] = df[v].where((df[v]>=lo)&(df[v]<=hi))
for v in RANGES:
    s=df[v]; med=s.median(); mad=(s-med).abs().median()
    if mad and mad>0: df[v]=s.where((s-med).abs()<=3.5*1.4826*mad)

harian = df.resample('D').mean().interpolate(method='time', limit=3, limit_area='inside').dropna(how='all')
print('Jumlah hari:', len(harian)); harian.tail()

## 2. Metrik & util (windowing + recursive multi-step)

In [ ]:
from sklearn.linear_model import LinearRegression
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING); logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

def mae(a,p):  a,p=np.asarray(a,float),np.asarray(p,float); return float(np.mean(np.abs(a-p)))
def rmse(a,p): a,p=np.asarray(a,float),np.asarray(p,float); return float(np.sqrt(np.mean((a-p)**2)))
def mape(a,p):
    a,p=np.asarray(a,float),np.asarray(p,float); m=np.abs(a)>1e-6
    return float(np.mean(np.abs((a[m]-p[m])/a[m]))*100) if m.sum() else float('nan')
def akurasi(a,p): return float(max(0.0, 100-mape(a,p)))

W = 7   # window (hari) untuk model lag
K = 10  # hari uji (holdout)

def windows(vals, W):
    X,y=[],[]
    for i in range(len(vals)-W): X.append(vals[i:i+W]); y.append(vals[i+W])
    return np.array(X), np.array(y)

def recursive_predict(step, last_window, H):
    win = last_window.astype('float32').copy(); out=[]
    for _ in range(H): yh=step(win); out.append(yh); win=np.append(win[1:], yh)
    return np.array(out)

def prophet_fit(dfser, sebelum=None):
    d = dfser.reset_index(); d.columns=['ds','y']
    if d['ds'].dt.tz is not None: d['ds']=d['ds'].dt.tz_localize(None)
    if sebelum is not None: d = d[d['ds'] < sebelum]
    m = Prophet(weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.01)
    m.fit(d); return m

## 3. Pilih model terbaik per variabel (multi-step holdout)

In [ ]:
VARS = ['suhu','udara','tanah']
hasil, terbaik = [], {}

for VAR in VARS:
    s = harian[VAR].dropna(); dates=list(s.index); vals=s.values.astype('float32')
    train, test_actual, test_dates = vals[:-K], vals[-K:], dates[-K:]
    Xtr, ytr = windows(train, W); lw = train[-W:]

    p_naive   = np.repeat(float(train[-1]), K)
    lr        = LinearRegression().fit(Xtr, ytr)
    p_lin     = recursive_predict(lambda w: float(lr.predict(w.reshape(1,-1))[0]), lw, K)
    mP        = prophet_fit(harian[[VAR]].dropna(), sebelum=test_dates[0])
    p_prophet = mP.predict(pd.DataFrame({'ds':test_dates}))['yhat'].values

    skor = {'Naive':p_naive, 'Linear':p_lin, 'Prophet':p_prophet}
    for n,p in skor.items():
        hasil.append({'variabel':VAR,'model':n,'MAE':round(mae(test_actual,p),3),'RMSE':round(rmse(test_actual,p),3),'Akurasi(%)':round(akurasi(test_actual,p),1)})
    terbaik[VAR] = min(skor, key=lambda n: mae(test_actual, skor[n]))

tabel = pd.DataFrame(hasil)
print(tabel.to_string(index=False))
print('\n>>> Model terbaik per variabel:', terbaik)

## 4. Grafik perbandingan

In [ ]:
import matplotlib.pyplot as plt
piv = tabel.pivot(index='variabel', columns='model', values='MAE')
piv.plot(kind='bar', figsize=(8,4), color={'Naive':'#9ca3af','Linear':'#a78bfa','Prophet':'#16a34a'})
plt.title('MAE per model (multi-step) - makin kecil makin baik'); plt.ylabel('MAE'); plt.grid(alpha=.3, axis='y'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
piv.round(3)

## 5. Hitung prediksi 14 hari (model terpilih) → model_config.json

In [ ]:
MAX_H = 14
CLAMP = {'suhu':(0,60), 'udara':(0,100), 'tanah':(0,100)}
def clamp(VAR, arr):
    lo,hi = CLAMP[VAR]; return [round(float(min(hi, max(lo, x))), 2) for x in arr]

config = {'info':{'dibuat':pd.Timestamp.now().isoformat(timespec='seconds'),
                  'last_date':harian.index.max().strftime('%Y-%m-%d'),
                  'max_horizon':MAX_H, 'satuan':{'suhu':'C','udara':'%','tanah':'%'},
                  'metode_evaluasi':'multi-step holdout (K=10)'},
          'variabel':{}}

for VAR in VARS:
    s = harian[VAR].dropna(); vals = s.values.astype('float32'); metode = terbaik[VAR]
    if metode == 'Prophet':
        m = prophet_fit(harian[[VAR]].dropna())
        pred = m.predict(m.make_future_dataframe(periods=MAX_H, freq='D')).tail(MAX_H)['yhat'].tolist()
    elif metode == 'Linear':
        Xf, yf = windows(vals, W); lr = LinearRegression().fit(Xf, yf)
        pred = recursive_predict(lambda w: float(lr.predict(w.reshape(1,-1))[0]), vals[-W:], MAX_H)
    else:  # Naive
        pred = [float(vals[-1])] * MAX_H
    config['variabel'][VAR] = {'metode': metode, 'prediksi14': clamp(VAR, pred)}
    print(f'{VAR:6s} -> {metode}')

with open('model_config.json','w') as f: json.dump(config, f, indent=2)
print('Tersimpan: model_config.json')
tabel.to_json('metrik_holdout.json', orient='records', indent=2)

try:
    from google.colab import files
    files.download('model_config.json'); files.download('metrik_holdout.json')
except Exception as e:
    print('Lewati auto-download:', e)

## 6. Langkah berikutnya

1. Upload `model_config.json` (baru) ke **HF Space** Anda, menimpa yang lama.
2. Space rebuild → website otomatis memakai mapping terbaru saat memilih sumber **Model HF**.

> Mapping kini selaras dengan analisis multi-step. Karena data terus bertambah, ulangi notebook ini
> secara berkala agar prediksi & pemilihan model tetap terbarui.